In [ ]:
import cv2
from matplotlib import pyplot as plt

img_path = r'screenshots\MuMu-20260701-205554-991.png'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig = plt.figure(frameon=False)
fig.set_size_inches(img_rgb.shape[1] / fig.dpi, img_rgb.shape[0] / fig.dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(img_rgb)
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0, 0)





In [ ]:
# 星星ROI
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 186, 279, 373, 466, 560, 653]
i, j = 0, 1
x1, y1 = 582 + x_offset[i], 287 + y_offset[j]
x2, y2 = 652 + x_offset[i], 304 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
import numpy as np

# 星级 ROI 去背景：HSV 提取金色星星，不裁剪底部
STAR_HSV_LOW = (20, 100, 130)
STAR_HSV_HIGH = (38, 255, 255)
STAR_MIN_AREA = 20  # 过滤 mask 内部的小噪点

x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 186, 279, 373, 466, 560, 653]
num_players = 8
num_heroes = 9


def isolate_stars(roi):
    """HSV 提取金色星星，返回 mask。"""
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, STAR_HSV_LOW, STAR_HSV_HIGH)
    kernel = np.ones((2, 2), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask


def count_stars_from_mask(mask, min_area=STAR_MIN_AREA):
    """统计 mask 中白色连通域数量，即星星数。"""
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    return sum(1 for i in range(1, n_labels) if stats[i, cv2.CC_STAT_AREA] >= min_area)


for j in range(num_players):
    stars = []
    for i in range(num_heroes):
        x1, y1 = 582 + x_offset[i], 287 + y_offset[j]
        x2, y2 = 652 + x_offset[i], 304 + y_offset[j]
        roi = img[y1:y2, x1:x2]
        stars.append(count_stars_from_mask(isolate_stars(roi)))
    while stars and stars[-1] == 0:
        stars.pop()
    print(f"玩家{j + 1}: " + " ".join(str(s) for s in stars))


In [ ]:
import cv2
import numpy as np
from pathlib import Path

# 模板文件夹
template_dir = Path(r'assets\templates\heroes')
num_players = 8    # 玩家数
num_heroes = 9     # 每位玩家最多9个英雄
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]

hero_templates = {}
empty_templates = {}
for path in template_dir.glob('*.jpg'):
    buf = np.frombuffer(path.read_bytes(), dtype=np.uint8)
    timg = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    hero_templates[path.name] = timg
print(f"已加载 {len(hero_templates)} 个英雄模板")

def crop_center(gray, margin_ratio=0.1):
    """裁掉边缘（绿框、底栏），只保留中间区域做匹配"""
    h, w = gray.shape[:2]
    mh, mw = int(h * margin_ratio), int(w * margin_ratio)
    return gray[mh:h - mh, mw:w - mw]

def is_empty_slot(roi, std_threshold=10, edge_threshold=0.05):
    """空槽为纯色绿底，几乎无纹理和边缘"""
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    return gray.std() < std_threshold or edges.mean() / 255 < edge_threshold

def match_roi_to_template(roi, templates, threshold=0.75, min_gap=0.08, padding=8, margin_ratio=0.1):
    """在 ROI 周围 padding 区域内滑动匹配，容忍坐标偏差。
    min_gap: 最高分与次高分之差，过滤空槽对多个模板的模糊匹配（如摩艾石像误报）"""
    roi_gray = crop_center(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), margin_ratio)
    search = cv2.copyMakeBorder(
        roi_gray, padding, padding, padding, padding, cv2.BORDER_REPLICATE
    )
    scores = []
    for name, timg in templates.items():
        temp_gray = crop_center(cv2.cvtColor(timg, cv2.COLOR_BGR2GRAY), margin_ratio)
        th, tw = temp_gray.shape
        if search.shape[0] < th or search.shape[1] < tw:
            continue
        res = cv2.matchTemplate(search, temp_gray, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, _ = cv2.minMaxLoc(res)
        scores.append((max_val, name))
    scores.sort(reverse=True)
    if not scores:
        return None, 0
    best_score, best_name = scores[0]
    second_score = scores[1][0] if len(scores) > 1 else 0
    if best_score >= threshold and (best_score - second_score) >= min_gap:
        return best_name, best_score
    return None, best_score

# 检测每位玩家每个格子，并记录匹配到的英雄名
detected_lineups = []
for j in range(num_players):
    print(f"玩家{j+1}：", end="")
    heroes = []
    for i in range(num_heroes):
        x1, y1 = 582 + x_offset[i], 306 + y_offset[j]
        x2, y2 = 652 + x_offset[i], 345 + y_offset[j]
        roi = img[y1:y2, x1:x2]
        if roi.shape[0] != (y2-y1) or roi.shape[1] != (x2-x1):
            break  # ROI 尺寸不符，直接断定本玩家只有i个英雄
        if is_empty_slot(roi):
            break  # 空槽，阵容结束
        tmp_name, score = match_roi_to_template(roi, hero_templates)
        if tmp_name is not None:
            heroes.append(tmp_name.replace('.jpg', ''))
        else:
            break  # 未匹配（含空槽），阵容结束
    detected_lineups.append(heroes)
    print("，".join(heroes) if heroes else "无英雄")

screenshot_filename = Path(img_path).name
print(f"\n确认无误后，运行下一个 cell 保存到 ground truth: {screenshot_filename}")

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

def add_ground_truth_entry(screenshot_filename, players_heroes, json_path='data/ground_truth.json'):
    """
    将截图和阵容加入到 ground_truth.json
    screenshot_filename: str, 例如 "MuMu-20260630-182408-786.png"
    players_heroes: list of list, 每一项为该玩家英雄名列表
    """
    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            ground_truth = json.load(f)
    else:
        ground_truth = {
            "version": 1,
            "description": "Hero lineup ground truth",
            "generated_at": datetime.now().astimezone().isoformat(),
            "detection_params": {},
            "template_count": 0,
            "screenshots": {},
        }

    if screenshot_filename in ground_truth["screenshots"]:
        print(f"警告: {screenshot_filename} 已存在，将覆盖原有数据。")

    ground_truth["screenshots"][screenshot_filename] = {
        "path": f"screenshots/{screenshot_filename}",
        "players": [
            {"player": idx + 1, "heroes": heroes}
            for idx, heroes in enumerate(players_heroes)
        ],
    }
    ground_truth["generated_at"] = datetime.now().astimezone().isoformat()
    ground_truth["template_count"] = len(hero_templates)
    ground_truth["detection_params"] = {
        "threshold": 0.75,
        "min_gap": 0.08,
        "padding": 8,
        "margin_ratio": 0.1,
        "empty_slot_std_threshold": 10,
        "empty_slot_edge_threshold": 0.05,
    }

    os.makedirs(os.path.dirname(json_path) or ".", exist_ok=True)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(ground_truth, f, ensure_ascii=False, indent=2)
        f.write("\n")

    print(f"已将 {screenshot_filename} 写入 {json_path}")

# 使用上一个 cell 的识别结果保存 ground truth
add_ground_truth_entry(screenshot_filename, detected_lineups)

In [ ]:
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 5, 3
x1, y1 = 582 + x_offset[i], 306 + y_offset[j]
x2, y2 = 652 + x_offset[i], 345 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
import numpy as np
from collections import Counter


def isolate_icon(roi, bg_color=(255, 255, 255), k=2):
    """保留前景图标，背景统一为单一颜色。前景用 kmeans 簇中心色，避免各行底板色污染抗锯齿边缘。"""
    h, w = roi.shape[:2]
    pixels = roi.reshape(-1, 3).astype(np.float32)

    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 10, cv2.KMEANS_PP_CENTERS
    )
    labels_2d = labels.reshape(h, w)

    # 边缘像素多数属于背景（图标居中、背景贴边）
    border = np.zeros((h, w), dtype=bool)
    border[0, :] = border[-1, :] = border[:, 0] = border[:, -1] = True
    bg_label = Counter(labels_2d[border].tolist()).most_common(1)[0][0]

    result = np.empty_like(roi)
    result[:] = bg_color
    for c in range(k):
        if c == bg_label:
            continue
        result[labels_2d == c] = centers[c].astype(np.uint8)
    return result


# 示例：任意 ROI 去背景
x_offset = [0, 77, 154]
y_offset = [0, 93, 187, 280, 373, 466, 560, 653]
i, j = 1, 5
x1, y1 = 1340 + x_offset[i], 305 + y_offset[j]
x2, y2 = 1385 + x_offset[i], 350 + y_offset[j]
roi = img[y1:y2, x1:x2]

icon = isolate_icon(roi, bg_color=(255, 255, 255))  # 背景可改成 (0, 0, 0) 等
icon_rgb = cv2.cvtColor(icon, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
axes[0].axis("off")
axes[1].imshow(icon_rgb)
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
x_offset = [0, 77, 154]
y_offset = [0, 93, 187, 280, 373, 466, 560, 653]
i, j = 0, 0
x1, y1 = 1340 + x_offset[i], 305 + y_offset[j]
x2, y2 = 1385 + x_offset[i], 350 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
import cv2
from matplotlib import pyplot as plt

img_path = r'screenshots\MuMu-20260701-181447-430.png'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig = plt.figure(frameon=False)
fig.set_size_inches(img_rgb.shape[1] / fig.dpi, img_rgb.shape[0] / fig.dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(img_rgb)
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0, 0)

In [ ]:
y_offset = [0, 93, 187, 280, 373, 466, 560, 653]
j = 7
x1, y1 = 520, 305 + y_offset[j]
x2, y2 = 580, 350 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
import numpy as np

# 阵容颜色 ROI（每位玩家左侧色块）
num_players = 8
y_offset = [0, 93, 187, 280, 373, 466, 560, 653]
COLOR_X1, COLOR_X2 = 520, 580
COLOR_Y_BASE, COLOR_H = 305, 45


def crop_team_color_roi(img, player: int) -> np.ndarray:
    y1 = COLOR_Y_BASE + y_offset[player]
    return img[y1 : y1 + COLOR_H, COLOR_X1:COLOR_X2]


def extract_team_color(roi: np.ndarray, bright_v: int = 230) -> np.ndarray:
    """提取代表色；过滤高光/过亮像素，用 LAB 中位数。"""
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = (hsv[:, :, 1] > 35) & (hsv[:, :, 2] > 50) & (hsv[:, :, 2] < bright_v)
    if mask.sum() < 15:
        mask = (hsv[:, :, 1] > 25) & (hsv[:, :, 2] > 40)
    if mask.sum() < 10:
        mask = np.ones(roi.shape[:2], dtype=bool)
    lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB)[mask]
    return np.median(lab, axis=0).astype(np.float32)


def color_distance(a: np.ndarray, b: np.ndarray, ignore_lightness: bool = True) -> float:
    if ignore_lightness:
        return float(np.linalg.norm(a[1:] - b[1:]))
    return float(np.linalg.norm(a - b))


def detect_highlight_player(colors, z_threshold: float = 1.2):
    """主页点进去时，该玩家 ROI 往往更亮。"""
    lightness = np.array([c[0] for c in colors], dtype=np.float32)
    med = float(np.median(lightness))
    mad = float(np.median(np.abs(lightness - med))) or 1.0
    scores = (lightness - med) / mad
    if scores.max() < z_threshold:
        return None
    return int(np.argmax(scores))


def _pair_players_subset(colors, players):
    if not players:
        return 0.0, []
    a = players[0]
    best = None
    for b in players[1:]:
        d = color_distance(colors[a], colors[b])
        rest_total, rest_pairs = _pair_players_subset(
            colors, [p for p in players if p not in (a, b)]
        )
        total = d + rest_total
        pairs = [(a, b)] + rest_pairs
        if best is None or total < best[0]:
            best = (total, pairs)
    return best


def pair_players_by_team_color(colors, highlight_player=None):
    """先为其余 6 人配出 3 队，剩余 2 人自动成队（抗单玩家高光）。"""
    n = len(colors)
    if highlight_player is None:
        highlight_player = detect_highlight_player(colors)

    if highlight_player is not None:
        remainder_candidates = [
            (min(highlight_player, j), max(highlight_player, j))
            for j in range(n)
            if j != highlight_player
        ]
    else:
        remainder_candidates = [(i, j) for i in range(n) for j in range(i + 1, n)]

    best = None
    for i, j in remainder_candidates:
        remaining = [p for p in range(n) if p not in (i, j)]
        total, inner_pairs = _pair_players_subset(colors, remaining)
        candidate = (total, (i, j), inner_pairs)
        if best is None or candidate[0] < best[0]:
            best = candidate

    _, remainder, inner_pairs = best
    pairs = inner_pairs + [remainder]
    return sorted((min(a, b), max(a, b)) for a, b in pairs), highlight_player


color_rois = [crop_team_color_roi(img, j) for j in range(num_players)]
team_colors = [extract_team_color(roi) for roi in color_rois]
pairs, highlight_idx = pair_players_by_team_color(team_colors)

print("玩家配对（同色为一队）：")
for k, (a, b) in enumerate(pairs, 1):
    tag = " ← 含高光玩家" if highlight_idx in (a, b) else ""
    print(f"  第{k}队: 玩家{a + 1} & 玩家{b + 1}{tag}")
if highlight_idx is not None:
    print(f"检测到可能受高光影响的玩家: 玩家{highlight_idx + 1}")

fig, axes = plt.subplots(2, 4, figsize=(10, 3))
for j, ax in enumerate(axes.ravel()):
    ax.imshow(cv2.cvtColor(color_rois[j], cv2.COLOR_BGR2RGB))
    partner = next((b + 1 if a == j else a + 1) for a, b in pairs if j in (a, b))
    title = f"P{j + 1} ↔ P{partner}"
    if j == highlight_idx:
        title += " *"
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import cv2
from matplotlib import pyplot as plt

img_path = r'screenshots\MuMu-20260701-230443-974.png'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig = plt.figure(frameon=False)
fig.set_size_inches(img_rgb.shape[1] / fig.dpi, img_rgb.shape[0] / fig.dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(img_rgb)
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0, 0)

In [ ]:
from pathlib import Path
import json

output_dir = Path(r'assets\templates\cards')
output_dir.mkdir(parents=True, exist_ok=True)

json_dir = Path(r'assets\templates')
json_dir.mkdir(parents=True, exist_ok=True)
json_path = json_dir / "card_icons.json"

# 8 位玩家 × 3 张卡，按截图从上到下填写卡牌名
card_matrix = [
    ["重质也重量pro", "核桃严选", "快速成型"],
    ["奇偶变换", "下雨了", "崇高牺牲pro"],
    ["攻防联合", "暴击专家pro", "守护"],
    ["满血才是王道", "超级卡包", "下雨了"],
    ["小而美pro·蓝", "天涯咫尺", "我吃年糕"],
    ["开攒", "起风了", "下雨了"],
    ["同舟共济", "起风了", "铜牌思维"],
    ["重质也重量pro", "物转法", "装备共鸣血"],
]

x_offset = [0, 77, 154]
y_offset = [0, 93, 187, 280, 373, 466, 560, 653]


def save_jpg(path: Path, image) -> None:
    """cv2.imwrite 在 Windows 下不支持中文路径，改用 imencode 写文件。"""
    ok, buf = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, 95])
    if not ok:
        raise RuntimeError(f"编码失败: {path}")
    path.write_bytes(buf.tobytes()
)

added = []
skipped = []
seen = set()

# 记录本次保存的卡牌及其截图
cur_screenshot = str(img_path)  # 当前图片路径

output_sample = {
    "screenshot": cur_screenshot,
    "card_matrix": card_matrix,
    # 删除icons键
}

for j, row in enumerate(card_matrix):
    for i, card_name in enumerate(row):
        card_name = card_name.strip()
        if not card_name:
            continue

        save_path = output_dir / f"{card_name}.jpg"
        if card_name in seen or save_path.exists():
            skipped.append(card_name)
            continue

        x1, y1 = 1340 + x_offset[i], 305 + y_offset[j]
        x2, y2 = 1385 + x_offset[i], 350 + y_offset[j]
        roi = img[y1:y2, x1:x2]
        icon_img = isolate_icon(roi, bg_color=(255, 255, 255))
        save_jpg(save_path, icon_img)
        seen.add(card_name)
        added.append(card_name)
        # 不再记录icons
        print(f"Added: {card_name}.jpg  (玩家{j + 1} 第{i + 1}张)")

print(f"\n本轮新增 {len(added)} 个模板: {', '.join(added) if added else '无'}")
if skipped:
    unique_skipped = list(dict.fromkeys(skipped))
    print(f"已跳过 {len(unique_skipped)} 个已有模板: {', '.join(unique_skipped)}")

# ---- 新的结构，维护多组截图-卡牌数据库 ----
# 数据结构: [
#   {
#     "screenshot": ...,
#     "card_matrix": ...
#   },
#   ...
# ]

prev_db = []
db = []

if json_path.exists():
    with open(json_path, "r", encoding="utf-8") as f:
        try:
            loaded = json.load(f)
            # 新结构：列表模式（或旧单一对象兼容）
            if isinstance(loaded, list):
                prev_db = loaded
            else:
                prev_db = [loaded]
        except Exception as e:
            print("卡牌数据库JSON解析失败，将覆盖写入")
            prev_db = []

screenshot_found = False
for sample in prev_db:
    if sample.get("screenshot", "") == cur_screenshot:
        # 覆盖已有的card_matrix
        sample.pop("icons", None)  # 删除旧的icons字段
        sample["card_matrix"] = output_sample["card_matrix"]
        sample["screenshot"] = output_sample["screenshot"]
        screenshot_found = True
        db.append(sample)
    else:
        db.append(sample)

if not screenshot_found:
    db.append(output_sample)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(db, f, ensure_ascii=False, indent=2)
print(f"模板元数据已写入: {json_path}")

In [ ]:
import cv2
import numpy as np
from collections import Counter
from pathlib import Path

template_dir = Path(r'assets\templates\cards')
num_players = 8
num_cards = 3
x_offset = [0, 77, 154]
y_offset = [0, 93, 187, 280, 373, 466, 560, 653]

# 形状 + 颜色联合匹配权重
SHAPE_WEIGHT = 0.55
COLOR_WEIGHT = 0.20
CHROMA_WEIGHT = 0.25
# 同形状不同色（如 卡牌宝袋 vs 最后的波纹·蓝）用颜色消歧
SHAPE_CLUSTER_THRESH = 0.8


def prepare_card_icon(roi, bg_color=(255, 255, 255), k=2):
    """去背景，返回 (icon, fg_mask)。"""
    h, w = roi.shape[:2]
    pixels = roi.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 10, cv2.KMEANS_PP_CENTERS
    )
    labels_2d = labels.reshape(h, w)
    border = np.zeros((h, w), dtype=bool)
    border[0, :] = border[-1, :] = border[:, 0] = border[:, -1] = True
    bg_label = Counter(labels_2d[border].tolist()).most_common(1)[0][0]

    icon = np.empty_like(roi)
    icon[:] = bg_color
    fg_mask = np.zeros((h, w), dtype=bool)
    for c in range(k):
        if c == bg_label:
            continue
        icon[labels_2d == c] = centers[c].astype(np.uint8)
        fg_mask |= labels_2d == c
    return icon, fg_mask


def crop_center(img, margin_ratio=0.1):
    h, w = img.shape[:2]
    mh, mw = int(h * margin_ratio), int(w * margin_ratio)
    return img[mh:h - mh, mw:w - mw]


def _template_match_score(search, template):
    th, tw = template.shape[:2]
    if search.shape[0] < th or search.shape[1] < tw:
        return 0.0
    res = cv2.matchTemplate(search, template, cv2.TM_CCOEFF_NORMED)
    return float(res.max())


def shape_score(roi_icon, tmpl_icon, padding=8, margin_ratio=0.1):
    roi_gray = crop_center(cv2.cvtColor(roi_icon, cv2.COLOR_BGR2GRAY), margin_ratio)
    tmpl_gray = crop_center(cv2.cvtColor(tmpl_icon, cv2.COLOR_BGR2GRAY), margin_ratio)
    search = cv2.copyMakeBorder(
        roi_gray, padding, padding, padding, padding, cv2.BORDER_REPLICATE
    )
    return _template_match_score(search, tmpl_gray)


def chroma_score(roi_icon, tmpl_icon, padding=8, margin_ratio=0.1):
    """LAB a*/b* 通道形状匹配，保留颜色分布差异。"""
    roi_lab = cv2.cvtColor(roi_icon, cv2.COLOR_BGR2LAB)
    tmpl_lab = cv2.cvtColor(tmpl_icon, cv2.COLOR_BGR2LAB)
    roi_ch = (
        crop_center(roi_lab[:, :, 1], margin_ratio).astype(np.float32)
        + crop_center(roi_lab[:, :, 2], margin_ratio).astype(np.float32)
    ) / 2
    tmpl_ch = (
        crop_center(tmpl_lab[:, :, 1], margin_ratio).astype(np.float32)
        + crop_center(tmpl_lab[:, :, 2], margin_ratio).astype(np.float32)
    ) / 2
    search = cv2.copyMakeBorder(
        roi_ch, padding, padding, padding, padding, cv2.BORDER_REPLICATE
    )
    return _template_match_score(search, tmpl_ch)


def color_score(roi_icon, tmpl_icon, roi_fg, tmpl_fg):
    """前景平均色相似度：LAB 色相 + HSV 饱和度。"""
    roi_lab = cv2.cvtColor(roi_icon, cv2.COLOR_BGR2LAB)
    tmpl_lab = cv2.cvtColor(tmpl_icon, cv2.COLOR_BGR2LAB)
    roi_hsv = cv2.cvtColor(roi_icon, cv2.COLOR_BGR2HSV)
    tmpl_hsv = cv2.cvtColor(tmpl_icon, cv2.COLOR_BGR2HSV)

    lab_dist = np.linalg.norm(
        roi_lab[roi_fg][:, 1:3].mean(axis=0) - tmpl_lab[tmpl_fg][:, 1:3].mean(axis=0)
    )
    lab_sim = max(0.0, 1.0 - lab_dist / 180.0)

    sat_dist = abs(
        roi_hsv[roi_fg][:, 1].mean() - tmpl_hsv[tmpl_fg][:, 1].mean()
    )
    sat_sim = max(0.0, 1.0 - sat_dist / 255.0)
    return lab_sim * 0.6 + sat_sim * 0.4


def combined_score(roi_icon, tmpl_icon, roi_fg, tmpl_fg, padding=8, margin_ratio=0.1):
    sh = shape_score(roi_icon, tmpl_icon, padding, margin_ratio)
    col = color_score(roi_icon, tmpl_icon, roi_fg, tmpl_fg)
    chroma = chroma_score(roi_icon, tmpl_icon, padding, margin_ratio)
    return SHAPE_WEIGHT * sh + COLOR_WEIGHT * col + CHROMA_WEIGHT * chroma


def adaptive_min_gap(best_score, min_gap=0.08):
    """高置信度时放宽 gap，避免同形状不同色卡牌被误拒。"""
    if best_score > 0.9:
        return 0.0
    # if best_score >= 0.98:
    #     return 0.02
    # if best_score >= 0.95:
    #     return 0.04
    return min_gap


card_template_sigs = {}
for path in template_dir.glob('*.jpg'):
    if path.name.startswith('player'):
        continue
    buf = np.frombuffer(path.read_bytes(), dtype=np.uint8)
    timg = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if timg is not None:
        icon, fg = prepare_card_icon(timg)
        card_template_sigs[path.name] = {"icon": icon, "fg": fg}
print(f"已加载 {len(card_template_sigs)} 个卡牌模板")


def match_card_roi(
    roi,
    template_sigs,
    threshold=0.75,
    min_gap=0.08,
    padding=8,
    margin_ratio=0.1,
):
    roi_icon, roi_fg = prepare_card_icon(roi)
    details = []
    for name, sig in template_sigs.items():
        sh = shape_score(roi_icon, sig["icon"], padding, margin_ratio)
        col = color_score(roi_icon, sig["icon"], roi_fg, sig["fg"])
        combined = combined_score(
            roi_icon, sig["icon"], roi_fg, sig["fg"], padding, margin_ratio
        )
        details.append(
            {"name": name, "combined": combined, "shape": sh, "color": col}
        )
    if not details:
        return "unknown", 0.0

    details.sort(key=lambda d: d["combined"], reverse=True)
    high_shape = [d for d in details if d["shape"] >= SHAPE_CLUSTER_THRESH]
    if len(high_shape) >= 2:
        color_ranked = sorted(high_shape, key=lambda d: d["color"], reverse=True)
        winner = color_ranked[0]
        second_color = color_ranked[1]["color"]
        if winner["color"] >= 0.85 and winner["color"] - second_color >= 0.03:
            return winner["name"].replace(".jpg", ""), winner["combined"]

    winner = details[0]
    second_score = details[1]["combined"] if len(details) > 1 else 0.0
    gap_threshold = adaptive_min_gap(winner["combined"], min_gap)
    if winner["combined"] >= threshold and (
        winner["combined"] - second_score
    ) >= gap_threshold:
        return winner["name"].replace(".jpg", ""), winner["combined"]
    return "unknown", winner["combined"]


detected_matrix = []
for j in range(num_players):
    print(f"玩家{j+1}：", end="")
    card_names = []
    for i in range(num_cards):
        x1, y1 = 1340 + x_offset[i], 305 + y_offset[j]
        x2, y2 = 1385 + x_offset[i], 350 + y_offset[j]
        roi = img[y1:y2, x1:x2]
        label, score = match_card_roi(roi, card_template_sigs)
        card_names.append(label)
    detected_matrix.append(card_names)
    print("，".join(card_names))

if detected_matrix == card_matrix:
    print("\n与 card_matrix 一致: 是")
else:
    print("\n与 card_matrix 一致: 否")
    for j, (detected_row, expected_row) in enumerate(zip(detected_matrix, card_matrix)):
        for i, (got, expected) in enumerate(zip(detected_row, expected_row)):
            if got != expected:
                print(f"  玩家{j + 1} 第{i + 1}张: 把 {expected} 误判为 {got}")

In [ ]:
# import os

# output_dir = r'assets\templates\heroes'
# os.makedirs(output_dir, exist_ok=True)

# num_players = 8    # j: 0-7 (共8位玩家)
# num_heroes = 9     # i: 0-8 (每位玩家最多9个英雄)
# x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
# y_offset = [0, 93, 185, 280, 372, 465, 559, 652]

# for j in range(num_players):
#     for i in range(num_heroes):
#         x1, y1 = 582 + x_offset[i], 306 + y_offset[j]
#         x2, y2 = 652 + x_offset[i], 345 + y_offset[j]
#         roi = img[y1:y2, x1:x2]
#         # 若roi尺寸不对，跳过
#         if roi.shape[0] != (y2-y1) or roi.shape[1] != (x2-x1):
#             print(f"skip invalid roi for player{j+1}_hero{i+1}")
#             continue
#         save_path = os.path.join(output_dir, f'player{j+1}_hero{i+1}.jpg')
#         cv2.imwrite(save_path, roi)
#         print(f"Saved: {save_path}")

In [ ]:
import cv2
from matplotlib import pyplot as plt

img_path = r'screenshots\MuMu-20260701-230443-974.png'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig = plt.figure(frameon=False)
fig.set_size_inches(img_rgb.shape[1] / fig.dpi, img_rgb.shape[0] / fig.dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(img_rgb)
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0, 0)

In [ ]:
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 3, 0
x1, y1 = 581 + x_offset[i], 345 + y_offset[j]
x2, y2 = 653 + x_offset[i], 369 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
# 装备区有三件装备时，第一件装备的ROI
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 0, 0
x1, y1 = 581 + x_offset[i], 345 + y_offset[j]
x2, y2 = 604 + x_offset[i], 369 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
# 装备区有三件装备时，第二件装备的ROI
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 0, 0
x1, y1 = 605 + x_offset[i], 345 + y_offset[j]
x2, y2 = 628 + x_offset[i], 369 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
# 装备区有三件装备时，第三件装备的ROI
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 0, 0
x1, y1 = 630 + x_offset[i], 345 + y_offset[j]
x2, y2 = 654 + x_offset[i], 369 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
# 装备区有两件装备时，第一件装备的ROI
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 3, 0
x1, y1 = 593 + x_offset[i], 345 + y_offset[j]
x2, y2 = 616 + x_offset[i], 369 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
# 装备区有两件装备时，第二件装备的ROI
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 3, 0
x1, y1 = 618 + x_offset[i], 345 + y_offset[j]
x2, y2 = 641 + x_offset[i], 369 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
# 装备区有一件装备时，装备的ROI
x_offset = [0, 74, 149, 223, 297, 371, 445, 519, 594]
y_offset = [0, 93, 185, 280, 372, 465, 559, 652]
i, j = 4, 0
x1, y1 = 606 + x_offset[i], 345 + y_offset[j]
x2, y2 = 629 + x_offset[i], 369 + y_offset[j]
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [ ]:
# 遍历截图：内联显示截图，并输出每位玩家英雄/星级/卡牌
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.detect_cards import detect_cards, load_template_sigs
from src.detect_heroes import detect_lineups, load_templates
from src.detect_stars import detect_stars
from src.parse import parse_hero_label

SCREENSHOT_DIR = ROOT / "screenshots"


def format_hero(label, stars):
    tier, hero_name = parse_hero_label(label)
    tier_text = f"{tier}费" if tier is not None else "?费"
    star_text = "★" * stars if stars > 0 else "0星"
    return f"{hero_name}({tier_text}, {star_text})"


def print_detection_result(img_path, img, hero_templates, card_sigs, index, total):
    lineups = detect_lineups(img, hero_templates)
    stars_by_player = detect_stars(img)
    cards_by_player = detect_cards(img, card_sigs)

    print(f"[{index + 1}/{total}] {img_path.name}")
    print("点“下一张”切换；按钮获得焦点后可按空格触发。")
    print("=" * 80)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    fig_width = 12
    fig_height = fig_width * h / w
    plt.figure(figsize=(fig_width, fig_height))
    plt.imshow(img_rgb)
    plt.axis("off")
    plt.show()

    for player_idx, (lineup, stars, card_row) in enumerate(
        zip(lineups, stars_by_player, cards_by_player), start=1
    ):
        hero_parts = []
        for hero in lineup["heroes"]:
            slot_index = hero["slot_index"]
            star_count = stars[slot_index] if slot_index < len(stars) else 0
            hero_parts.append(format_hero(hero["label"], star_count))

        card_parts = [card["label"] for card in card_row["cards"]]
        print(f"玩家{player_idx} / 排名{player_idx}")
        print("  英雄：")
        if hero_parts:
            for hero_order, hero_text in enumerate(hero_parts, start=1):
                print(f"    {hero_order}. {hero_text}")
        else:
            print("    未识别")
        print("  卡牌：")
        for card_order, card_text in enumerate(card_parts, start=1):
            print(f"    {card_order}. {card_text}")
        print()


def render_current():
    output = state["output"]
    index = state["index"]
    img_path = screenshots[index]
    img = cv2.imread(str(img_path))

    with output:
        clear_output(wait=True)
        if img is None:
            print(f"读取失败：{img_path}")
            return
        print_detection_result(
            img_path,
            img,
            hero_templates,
            card_sigs,
            index,
            len(screenshots),
        )


def go_next(_button=None):
    if state["index"] < len(screenshots) - 1:
        state["index"] += 1
    render_current()


def go_prev(_button=None):
    if state["index"] > 0:
        state["index"] -= 1
    render_current()


screenshots = sorted(SCREENSHOT_DIR.glob("*.png"))
if not screenshots:
    print(f"未找到截图：{SCREENSHOT_DIR}")
else:
    print("Loading templates...")
    hero_templates = load_templates()
    card_sigs = load_template_sigs()
    print(f"heroes={len(hero_templates)}, cards={len(card_sigs)}")

    if widgets is None:
        print("当前环境没有 ipywidgets，将一次性输出第一张截图。")
        img = cv2.imread(str(screenshots[0]))
        if img is not None:
            print_detection_result(screenshots[0], img, hero_templates, card_sigs, 0, len(screenshots))
    else:
        prev_button = widgets.Button(description="上一张")
        next_button = widgets.Button(description="下一张（可按空格）", button_style="primary")
        state = {"index": 0, "output": widgets.Output()}

        prev_button.on_click(go_prev)
        next_button.on_click(go_next)

        display(widgets.HBox([prev_button, next_button]))
        display(state["output"])
        render_current()


In [ ]:
import cv2
from matplotlib import pyplot as plt

img_path = r'screenshots.equipments\微信图片_20260702162659_10_63.jpg'
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig = plt.figure(frameon=False)
fig.set_size_inches(img_rgb.shape[1] / fig.dpi, img_rgb.shape[0] / fig.dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(img_rgb)
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.margins(0, 0)

In [ ]:
# 前7张截图
i, j = 0, 0
x1, y1 = 210 + 138 * i, 257 + 137 * j
x2, y2 = 330 + 138 * i, 374 + 137 * j
roi = img[y1:y2, x1:x2]
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
plt.imshow(roi_rgb)
plt.axis('off')
plt.show()

In [107]:
from pathlib import Path

import cv2
import numpy as np

# 批量提取前 7 张截图的装备模板 ROI
screenshot_paths = [
    # Path(r"screenshots.equipments\微信图片_20260702162659_10_63.jpg"),
    # Path(r"screenshots.equipments\微信图片_20260702162701_11_63.jpg"),
    # Path(r"screenshots.equipments\微信图片_20260702162703_12_63.jpg"),
    # Path(r"screenshots.equipments\微信图片_20260702162704_13_63.jpg"),
    # Path(r"screenshots.equipments\微信图片_20260702162706_14_63.jpg"),
    # Path(r"screenshots.equipments\微信图片_20260702162707_15_63.jpg"),
    # Path(r"screenshots.equipments\微信图片_20260702162708_16_63.jpg"),
    Path(r"screenshots.equipments\微信图片_20260702162711_17_63.jpg"),
]

output_dir = Path(r"assets\templates\equipments")
output_dir.mkdir(parents=True, exist_ok=True)

num_rows = 5
num_cols = 3
roi_w = 330 - 210
roi_h = 374 - 257

saved = []
for screenshot_path in screenshot_paths:
    img = cv2.imdecode(np.frombuffer(screenshot_path.read_bytes(), dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"无法读取截图: {screenshot_path}")

    for row in range(num_rows):
        for col in range(num_cols):
            x1, y1 = 210 + 138 * col, 342 + 137 * row
            x2, y2 = x1 + roi_w, y1 + roi_h
            roi = img[y1:y2, x1:x2]

            if roi.shape[:2] != (roi_h, roi_w):
                raise ValueError(
                    f"ROI 超出图片范围: {screenshot_path.name}, row={row}, col={col}, "
                    f"shape={roi.shape[:2]}"
                )

            out_path = output_dir / f"{screenshot_path.stem}_r{row + 1:02d}_c{col + 1:02d}.jpg"
            ok, encoded = cv2.imencode(".jpg", roi)
            if not ok:
                raise ValueError(f"无法编码 ROI: {out_path}")
            out_path.write_bytes(encoded.tobytes())
            saved.append(out_path)

print(f"已保存 {len(saved)} 个装备模板 ROI 到 {output_dir}")
print("示例:")
for path in saved[:5]:
    print(path)

已保存 15 个装备模板 ROI 到 assets\templates\equipments
示例:
assets\templates\equipments\微信图片_20260702162711_17_63_r01_c01.jpg
assets\templates\equipments\微信图片_20260702162711_17_63_r01_c02.jpg
assets\templates\equipments\微信图片_20260702162711_17_63_r01_c03.jpg
assets\templates\equipments\微信图片_20260702162711_17_63_r02_c01.jpg
assets\templates\equipments\微信图片_20260702162711_17_63_r02_c02.jpg
